<a href="https://colab.research.google.com/github/Nancy-Batra/Flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nancy-Batra/Flyrank-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Answer:** My lane is a **ranking/scoring** problem. The goal of this lane is to give each content page a priority score and ranking so that the team can review the most important pages first. Ranking is appropriate because the team has limited capacity and needs to decide which pages should be reviewed first.

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Task type: Ranking / Scoring")
print("Number of pages:", len(df))

Task type: Ranking / Scoring
Number of pages: 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Answer:** For the starter dataset, I will use 'is_declining_label' as a proxy target. When its 'trend_direction' is 'down', the page is labeled 1 and 0 otherwise. It is only a starter proxy rather than an ideal future looking outcome.

In [14]:
df["is_declining_label"] = ( df["trend_direction"].str.lower() == "down").astype(int)
print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Answer**: I will use Precision@50 as the main success metric. It measures how many of the top 50 ranked pages are actually positive according to the proxy target. This matches the real decision because the content team has limited time and may only be able to review a fixed number of pages.

In [15]:
def precision_at_k(scores, labels, k):
    order = scores.argsort()[::-1]
    top_k = labels.iloc[order[:k]]
    return top_k.mean()

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Answer**: The unit of analysis is one content page. Each row represents one page and contains signals such as impressions, CTR, average position, content age, and freshness information.

In [16]:
page_sample = df[
    [
        "content_id",
        "impressions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "days_since_last_update"
    ]
]

page_sample.head(10)


,content_id,impressions_90d,ctr,avg_position,content_age_days,days_since_last_update
0,content_304f48230142,3803,0.76,10.6,187,20
1,content_a1fb4e703a9e,15320,0.05,20.3,445,25
2,content_9aa793d4d895,12581,0.09,36.5,141,20
3,content_331d6c4de07b,11751,0.49,6.2,463,22
4,content_d99b7a2d90ca,19140,0.13,44.0,263,14
5,content_d4084a4bc775,3970,0.03,8.5,147,20
6,content_9a34b442b552,20,0.00,7.0,90,20
7,content_a63219c6e95a,1724,0.06,21.2,445,22
8,content_5e6c160719bc,32574,0.09,46.0,90,20
9,content_c27558df2b0c,1240,0.16,4.9,257,104


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Answer**: A fixed rule can rank pages using simple conditions, such as selecting pages that are both stale and visible but page performance can depend on several signals such as; impressions, CTR, average position, content age and engagement.

ML can identify patterns that may be difficult to describe with one simple if-statement.

In [17]:
features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

print("Signals available for ranking:", features)

Signals available for ranking: ['impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'word_count']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.